<hr style="
    border: 0;
    height: 2px;
    background-color:#777777;
    width:65%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">


<h1 style="text-align:center; color:#003366; font-size:48px;">
PROYECTO FINAL
</h1>

<hr style="
    border: 0;
    height: 2px;
    background-color:GOLD;
    width:60%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:GOLD; font-size:28px;">
Europe countries vs Target 2030

</h2>

In [16]:
# --- Core ---
import warnings
warnings.filterwarnings("ignore")

# --- Standard library ---
import os
import json
import time
from datetime import datetime, date

# --- Data handling ---
import pandas as pd
import numpy as np

# --- Modeling ---
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.metrics import mean_absolute_error
from scipy.optimize import curve_fit
from scipy.stats import pearsonr


# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go


# --- Notebook ---
%matplotlib inline

# --- Logging cleanup for Prophet ---
import logging
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)


In [17]:
# Save_df
def save_df(df, name, folder):
    path = fr"{folder}\{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")

# Metrics target ember

metrics_target = [
    "hydro_capacity_gw", "renewables_share_elec", "solar_capacity_gw",
    "wind_capacity_gw", "low_carbon_share_elec", "renewables_capacity_gw",
    "fossil_capacity_gw", "fossil_electricity", "fossil_share_elec",
    "hydro_electricity", "hydro_share_elec", "nuclear_capacity_gw",
    "nuclear_electricity", "nuclear_share_elec", "coal_capacity_gw",
    "coal_electricity", "coal_share_elec", "gas_capacity_gw",
    "gas_electricity", "gas_share_elec", "solar_electricity",
    "solar_share_elec", "wind_electricity", "wind_share_elec",
    "biofuel_capacity_gw", "biofuel_electricity", "biofuel_share_elec",
    "renewables_electricity", "renewables_share_energy"]


columns = ["iso_code","country","year"] + metrics_target

In [18]:
# Import OWID table
Country_Owid = pd.read_csv(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\dfs_global\Country_Owid.csv")


# Filter country with data in 2025, are the european ones, i have ready check
year_2025 = Country_Owid[Country_Owid["year"] == 2025]

Europe_2025 = Country_Owid[(Country_Owid["country"].isin(year_2025["country"]))&
                           (Country_Owid["iso_code"].notna())&
                           (Country_Owid["year"].between(2000,2025))]

Europe_2025 = Europe_2025.loc[:, Europe_2025.columns.isin(columns)]


# Check
Europe_2025.info()
pd.set_option("display.max_columns", None)
Europe_2025.tail()

<class 'pandas.core.frame.DataFrame'>
Index: 905 entries, 1862 to 21732
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   country                  905 non-null    object 
 1   year                     905 non-null    int64  
 2   iso_code                 905 non-null    object 
 3   biofuel_electricity      888 non-null    float64
 4   biofuel_share_elec       888 non-null    float64
 5   coal_electricity         905 non-null    float64
 6   coal_share_elec          905 non-null    float64
 7   fossil_electricity       905 non-null    float64
 8   fossil_share_elec        905 non-null    float64
 9   gas_electricity          893 non-null    float64
 10  gas_share_elec           893 non-null    float64
 11  hydro_electricity        905 non-null    float64
 12  hydro_share_elec         905 non-null    float64
 13  low_carbon_share_elec    905 non-null    float64
 14  nuclear_electricity      8

,country,year,iso_code,biofuel_electricity,biofuel_share_elec,coal_electricity,coal_share_elec,fossil_electricity,fossil_share_elec,gas_electricity,gas_share_elec,hydro_electricity,hydro_share_elec,low_carbon_share_elec,nuclear_electricity,nuclear_share_elec,renewables_electricity,renewables_share_elec,renewables_share_energy,solar_electricity,solar_share_elec,wind_electricity,wind_share_elec
21728,United Kingdom,2021,GBR,40.009998,12.994478,6.79,2.205261,139.309998,45.245209,122.839996,39.896069,5.42,1.760312,54.754791,46.099998,14.972393,122.489998,39.782398,17.482069,12.13,3.939591,64.930000,21.088017
21729,United Kingdom,2022,GBR,35.849998,11.045043,5.94,1.830057,142.119995,43.785816,125.110001,38.545200,5.66,1.743792,56.214188,47.400002,14.603489,135.059998,41.610699,19.072384,13.34,4.109927,80.209999,24.711937
21730,United Kingdom,2023,GBR,34.099998,11.656923,3.78,1.292175,116.260002,39.742931,101.660004,34.751991,5.54,1.893823,60.257069,40.599998,13.878918,135.669998,46.378147,19.976458,13.88,4.744812,82.150002,28.082590
21731,United Kingdom,2024,GBR,40.090000,14.108249,1.90,0.668637,99.650002,35.068272,86.300003,30.370214,5.76,2.027027,64.931725,40.590000,14.284206,143.919998,50.647522,21.292688,14.79,5.204814,83.279999,29.307432
21732,United Kingdom,2025,GBR,41.209999,14.075415,0.33,0.112713,104.010002,35.524967,90.910004,31.050619,5.55,1.895621,64.475037,36.380001,12.425713,152.389999,52.049320,NaN,19.32,6.598811,86.309998,29.479471


<hr style="
    border: 0;
    height: 2px;
    background-color:GOLD;
    width:60%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:GOLD; font-size:28px;">
New Model Linear Regression
</h2>

In [19]:
def predict_and_fill_europe_linear(df_long):
    # Fonti di base assolute
    base_sources = ['hydro', 'solar', 'wind', 'biofuel', 'nuclear', 'coal', 'gas']
    all_years = np.arange(2000, 2031) 
    countries = df_long[['iso_code', 'country']].drop_duplicates()
    
    print(f"Start(Gap-Filling & Forecasting) for {len(countries)} countries...")
    
    all_rows = []
    
    for _, c_row in countries.iterrows():
        iso = c_row['iso_code']
        c_name = c_row['country']
        
        df_c = df_long[df_long['iso_code'] == iso].copy()
        country_data = {}
        
       # -----------------------------------------------------------------
       # PHASE 1: LINEAR REGRESSION TO FILL GAPS AND PREDICT FUTURE VALUES
       #  -----------------------------------------------------------------

        vars_to_predict = [f"{s}_electricity" for s in base_sources] + ["renewables_share_energy"]
        
        for var in vars_to_predict:
            if var not in df_c.columns:
                country_data[var] = np.full(len(all_years), np.nan)
                continue
                
            full_series = np.full(len(all_years), np.nan)
            
            for _, row in df_c.iterrows():
                y = int(row['year'])
                if 2000 <= y <= 2030:
                    full_series[y - 2000] = row[var]
            
            mask = ~np.isnan(full_series)
            if mask.sum() < 3: 
                country_data[var] = full_series 
                continue
                
            y_train = full_series[mask]
            X_train = all_years[mask].reshape(-1, 1)
            X_pred = all_years.reshape(-1, 1)
            
            # Rules linear
            if "solar" in var or "wind" in var:
                recent_n = min(7, len(y_train)) # last 7 years
                model = LinearRegression().fit(X_train[-recent_n:], y_train[-recent_n:])
                pred = model.predict(X_pred)
                pred = np.maximum(pred, 0)
                
            elif "hydro" in var:
                pred = np.full(len(all_years), np.nanmean(y_train[-5:]))
                
            else:
                model = LinearRegression().fit(X_train, y_train)
                pred = model.predict(X_pred)
                pred = np.maximum(pred, 0)
                if "share" in var: pred = np.minimum(pred, 100)
                
            # MAGIC: It overwrites only the NaN values (the gaps) with the prediction!
            full_series = np.where(np.isnan(full_series), pred, full_series)
            country_data[var] = full_series
            
        # -----------------------------------------------------------------
        # PHASE 2: RECALCULATION OF ELECTRICITY SHARES (100% Accurate)
        # -----------------------------------------------------------------

        def safe_sum_series(keys):
            arrays = [country_data[k] for k in keys if k in country_data]
            if not arrays: return np.full(len(all_years), np.nan)
            return np.nansum(np.array(arrays), axis=0)
            
        country_data["renewables_electricity"] = safe_sum_series([f"{s}_electricity" for s in ["hydro", "solar", "wind", "biofuel"]])
        country_data["fossil_electricity"] = safe_sum_series([f"{s}_electricity" for s in ["coal", "gas"]])
        
        nuc_elec = country_data.get("nuclear_electricity", np.zeros(len(all_years)))
        nuc_elec = np.where(np.isnan(nuc_elec), 0, nuc_elec)
        country_data["low_carbon_electricity"] = country_data["renewables_electricity"] + nuc_elec
        
        total_elec = safe_sum_series([f"{s}_electricity" for s in base_sources])
        safe_total = np.where(total_elec == 0, np.nan, total_elec)
        
        for source in base_sources:
            if f"{source}_electricity" in country_data:
                country_data[f"{source}_share_elec"] = (country_data[f"{source}_electricity"] / safe_total) * 100
                
        country_data["renewables_share_elec"] = (country_data["renewables_electricity"] / safe_total) * 100
        country_data["fossil_share_elec"] = (country_data["fossil_electricity"] / safe_total) * 100
        country_data["low_carbon_share_elec"] = (country_data["low_carbon_electricity"] / safe_total) * 100
        
        # FALLBACK
        if "renewables_share_energy" in country_data:
            mask_nan = np.isnan(country_data["renewables_share_energy"])
            country_data["renewables_share_energy"][mask_nan] = country_data["renewables_share_elec"][mask_nan]
            
        # -----------------------------------------------------------------
        # PHASE 3: REBUILD DF
        # -----------------------------------------------------------------
        for i, year in enumerate(all_years):
            row_dict = {'country': c_name, 'year': year, 'iso_code': iso}
            for col in df_long.columns:
                if col not in ['country', 'year', 'iso_code']:
                    row_dict[col] = country_data.get(col, np.full(len(all_years), np.nan))[i]
            all_rows.append(row_dict)
            
    df_final = pd.DataFrame(all_rows)
    print("Processo completato! Il Gap-Filling ha avuto successo.")
    return df_final.round(3)


In [20]:
# RUN:
df_europa_completo = predict_and_fill_europe_linear(Europe_2025)
display(df_europa_completo[(df_europa_completo['iso_code'] == 'ITA') & (df_europa_completo['year'] >= 2020)])

Start(Gap-Filling & Forecasting) for 35 countries...
Processo completato! Il Gap-Filling ha avuto successo.


,country,year,iso_code,biofuel_electricity,biofuel_share_elec,coal_electricity,coal_share_elec,fossil_electricity,fossil_share_elec,gas_electricity,gas_share_elec,hydro_electricity,hydro_share_elec,low_carbon_share_elec,nuclear_electricity,nuclear_share_elec,renewables_electricity,renewables_share_elec,renewables_share_energy,solar_electricity,solar_share_elec,wind_electricity,wind_share_elec
485,Italy,2020,ITA,19.620,7.606,13.380,5.187,147.080,57.019,133.700,51.832,47.550,18.434,42.981,0.0,0.0,110.870,42.981,19.399,24.940,9.669,18.760,7.273
486,Italy,2021,ITA,19.060,7.100,14.020,5.223,158.030,58.868,144.010,53.645,45.390,16.908,41.132,0.0,0.0,110.420,41.132,18.014,25.040,9.328,20.930,7.797
487,Italy,2022,ITA,17.610,6.807,22.610,8.740,164.080,63.425,141.470,54.685,28.400,10.978,36.575,0.0,0.0,94.620,36.575,15.951,28.120,10.870,20.490,7.920
488,Italy,2023,ITA,16.010,6.588,13.220,5.440,132.240,54.415,119.020,48.975,40.520,16.674,45.585,0.0,0.0,110.780,45.585,18.854,30.710,12.637,23.540,9.686
489,Italy,2024,ITA,17.230,6.863,3.950,1.573,122.540,48.809,118.590,47.236,53.130,21.162,51.191,0.0,0.0,128.520,51.191,20.931,35.990,14.335,22.170,8.831
490,Italy,2025,ITA,15.610,6.201,3.730,1.482,128.780,51.160,125.050,49.678,41.270,16.395,48.840,0.0,0.0,122.940,48.840,21.367,44.630,17.730,21.430,8.513
491,Italy,2026,ITA,22.915,8.201,15.945,5.706,148.425,53.120,132.481,47.413,41.742,14.939,46.880,0.0,0.0,130.992,46.880,22.031,43.387,15.528,22.949,8.213
492,Italy,2027,ITA,23.721,8.388,14.720,5.205,147.296,52.085,132.576,46.880,41.742,14.760,47.915,0.0,0.0,135.503,47.915,22.695,46.623,16.486,23.417,8.280
493,Italy,2028,ITA,24.527,8.571,13.496,4.716,146.167,51.075,132.671,46.359,41.742,14.586,48.925,0.0,0.0,140.013,48.925,23.359,49.858,17.422,23.886,8.346
494,Italy,2029,ITA,25.334,8.749,12.271,4.238,145.038,50.089,132.767,45.851,41.742,14.416,49.911,0.0,0.0,144.523,49.911,24.023,53.093,18.336,24.354,8.411


In [21]:
# 1. Select only numeric columns
value_cols = [
    col for col in df_europa_completo.columns
    if col not in ["country", "iso_code", "year"]]

# Convert everything to numeric (ignore errors)
df_europa_completo[value_cols] = df_europa_completo[value_cols].apply(pd.to_numeric, errors="coerce")

# 2. Melt → long format
df_long = df_europa_completo.melt(
    id_vars=["country", "iso_code", "year"],
    value_vars=value_cols,
    var_name="variable",
    value_name="value")

# 3. Pivot → country × variable × year
df_pivoted = df_long.pivot_table(
    index=["country", "iso_code", "variable"],
    columns="year",
    values="value",
    aggfunc="mean").reset_index()

# 4. Round values
df_pivoted = df_pivoted.round(2)

# 5.Rename variable
df_pivoted = df_pivoted.rename(columns={"variable":"energy_metric"})

print("Unique countries:", df_pivoted["iso_code"].nunique())
df_pivoted.head(10)

Unique countries: 35


year,country,iso_code,energy_metric,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030
0,Austria,AUT,biofuel_electricity,1.53,1.67,1.52,1.65,1.91,2.43,3.23,4.06,4.16,4.28,4.48,4.56,4.68,4.70,4.55,4.65,4.78,4.92,4.93,4.66,4.59,4.48,4.75,4.56,4.67,4.51,5.64,5.77,5.90,6.03,6.16
1,Austria,AUT,biofuel_share_elec,2.68,2.88,2.63,3.01,3.27,3.98,5.50,6.80,6.79,6.76,6.97,7.76,7.15,7.66,7.78,7.99,7.76,7.72,7.98,6.91,6.96,7.07,7.75,6.71,6.18,6.31,7.52,7.50,7.48,7.46,7.44
2,Austria,AUT,coal_electricity,5.72,6.89,6.61,8.44,7.91,7.17,7.37,6.60,5.53,3.76,4.92,5.43,4.39,4.21,2.94,2.95,2.04,1.76,1.81,1.50,0.56,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,Austria,AUT,coal_share_elec,10.03,11.90,11.43,15.39,13.53,11.74,12.56,11.06,9.03,5.94,7.66,9.24,6.71,6.86,5.03,5.07,3.31,2.76,2.93,2.22,0.85,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,Austria,AUT,fossil_electricity,13.57,15.64,15.92,19.60,18.86,20.20,18.04,16.50,16.71,16.10,19.27,17.87,14.10,10.86,8.26,10.61,10.53,12.67,11.73,12.82,10.51,10.62,10.85,7.50,7.54,8.63,9.04,8.98,8.92,8.85,8.79
5,Austria,AUT,fossil_share_elec,23.80,27.02,27.53,35.74,32.25,33.07,30.73,27.65,27.29,25.44,29.99,30.40,21.55,17.70,14.13,18.23,17.09,19.88,18.98,19.00,15.94,16.76,17.71,11.04,9.99,12.07,12.05,11.66,11.29,10.94,10.60
6,Austria,AUT,gas_electricity,7.85,8.75,9.31,11.16,10.95,13.03,10.67,9.90,11.18,12.34,14.35,12.44,9.71,6.65,5.32,7.66,8.49,10.91,9.92,11.32,9.95,10.62,10.85,7.50,7.54,8.63,9.04,8.98,8.92,8.85,8.79
7,Austria,AUT,gas_share_elec,13.77,15.12,16.10,20.35,18.72,21.33,18.18,16.59,18.26,19.50,22.33,21.16,14.84,10.84,9.10,13.16,13.78,17.12,16.05,16.78,15.09,16.76,17.71,11.04,9.99,12.07,12.05,11.66,11.29,10.94,10.60
8,Austria,AUT,hydro_electricity,41.84,40.46,40.23,33.21,36.76,37.10,35.66,37.05,38.33,40.90,38.36,34.24,43.85,42.02,41.01,37.16,39.97,38.29,37.64,40.83,42.00,38.75,34.63,40.67,45.71,39.16,39.78,39.78,39.78,39.78,39.78
9,Austria,AUT,hydro_share_elec,73.39,69.90,69.58,60.56,62.86,60.74,60.75,62.09,62.59,64.63,59.70,58.25,67.02,68.48,70.15,63.85,64.88,60.09,60.92,60.52,63.70,61.15,56.52,59.84,60.54,54.78,53.03,51.68,50.39,49.16,47.99



<hr style="
    border: 0;
    height: 2px;
    background-color:GOLD;
    width:60%;
    margin:25px auto;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<h2 style="text-align:center; color:GOLD; font-size:28px;">
EMBER target 2030
</h2>

In [22]:
# Load target ember 2030 

path = r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\dfs_global\Ember_2030.csv"
Ember_2030 = pd.read_csv(path)

# Check
Ember_2030.head()

,iso_code,target_year,fuel_category,metric,collection_year,2030_target,variable
0,ALB,2030,Hydro,capacity_total_gw,2024,2.51,hydro_capacity_gw
1,ALB,2030,Renewables,share_of_generation_pct,2024,100.00,renewables_share_elec
2,ALB,2030,Solar,capacity_total_gw,2024,0.49,solar_capacity_gw
3,ALB,2030,Wind,capacity_total_gw,2024,0.30,wind_capacity_gw
4,ARE,2030,Clean,share_of_generation_pct,2023,32.00,low_carbon_share_elec


In [23]:
# 1. KEEP OLDEST collection_year 
ember = (
    Ember_2030
    .sort_values("collection_year")
    .drop_duplicates(
        subset=["iso_code", "target_year", "fuel_category", "metric"],
        keep="first"))

# 3. DROP CAPACITY METRICS
ember = ember[~ember["variable"].str.contains("capacity")]
ember = ember.rename(columns={"variable":"energy_metric"})

ember

,iso_code,target_year,fuel_category,metric,collection_year,2030_target,energy_metric
16,AUS,2030,Coal,generation_twh,2023,40.70,coal_electricity
17,AUS,2030,Coal,share_of_generation_pct,2023,16.60,coal_share_elec
19,AUS,2030,Gas,generation_twh,2023,5.50,gas_electricity
20,AUS,2030,Gas,share_of_generation_pct,2023,2.20,gas_share_elec
22,AUS,2030,Hydro,generation_twh,2023,15.30,hydro_electricity
...,...,...,...,...,...,...,...
212,CZE,2030,Solar,generation_twh,2026,9.91,solar_electricity
214,CZE,2030,Wind,generation_twh,2026,3.93,wind_electricity
202,CZE,2030,Bioenergy,generation_twh,2026,5.61,biofuel_electricity
294,EST,2030,Renewables,share_of_generation_pct,2026,0.90,renewables_share_elec


In [24]:
# Merge europe and ember
df_europe_target = pd.merge(df_pivoted,ember[["iso_code", "energy_metric", "2030_target"]],  
    on=["iso_code", "energy_metric"],how="left")

# Keep target not NaN
df_europe_target = df_europe_target[df_europe_target["2030_target"].notna()].round(2)

# Check
df_europe_target.head(40)

,country,iso_code,energy_metric,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,2030_target
0,Austria,AUT,biofuel_electricity,1.53,1.67,1.52,1.65,1.91,2.43,3.23,4.06,4.16,4.28,4.48,4.56,4.68,4.70,4.55,4.65,4.78,4.92,4.93,4.66,4.59,4.48,4.75,4.56,4.67,4.51,5.64,5.77,5.90,6.03,6.16,6.00
1,Austria,AUT,biofuel_share_elec,2.68,2.88,2.63,3.01,3.27,3.98,5.50,6.80,6.79,6.76,6.97,7.76,7.15,7.66,7.78,7.99,7.76,7.72,7.98,6.91,6.96,7.07,7.75,6.71,6.18,6.31,7.52,7.50,7.48,7.46,7.44,6.12
4,Austria,AUT,fossil_electricity,13.57,15.64,15.92,19.60,18.86,20.20,18.04,16.50,16.71,16.10,19.27,17.87,14.10,10.86,8.26,10.61,10.53,12.67,11.73,12.82,10.51,10.62,10.85,7.50,7.54,8.63,9.04,8.98,8.92,8.85,8.79,6.00
5,Austria,AUT,fossil_share_elec,23.80,27.02,27.53,35.74,32.25,33.07,30.73,27.65,27.29,25.44,29.99,30.40,21.55,17.70,14.13,18.23,17.09,19.88,18.98,19.00,15.94,16.76,17.71,11.04,9.99,12.07,12.05,11.66,11.29,10.94,10.60,6.20
6,Austria,AUT,gas_electricity,7.85,8.75,9.31,11.16,10.95,13.03,10.67,9.90,11.18,12.34,14.35,12.44,9.71,6.65,5.32,7.66,8.49,10.91,9.92,11.32,9.95,10.62,10.85,7.50,7.54,8.63,9.04,8.98,8.92,8.85,8.79,7.00
7,Austria,AUT,gas_share_elec,13.77,15.12,16.10,20.35,18.72,21.33,18.18,16.59,18.26,19.50,22.33,21.16,14.84,10.84,9.10,13.16,13.78,17.12,16.05,16.78,15.09,16.76,17.71,11.04,9.99,12.07,12.05,11.66,11.29,10.94,10.60,7.14
8,Austria,AUT,hydro_electricity,41.84,40.46,40.23,33.21,36.76,37.10,35.66,37.05,38.33,40.90,38.36,34.24,43.85,42.02,41.01,37.16,39.97,38.29,37.64,40.83,42.00,38.75,34.63,40.67,45.71,39.16,39.78,39.78,39.78,39.78,39.78,47.00
9,Austria,AUT,hydro_share_elec,73.39,69.90,69.58,60.56,62.86,60.74,60.75,62.09,62.59,64.63,59.70,58.25,67.02,68.48,70.15,63.85,64.88,60.09,60.92,60.52,63.70,61.15,56.52,59.84,60.54,54.78,53.03,51.68,50.39,49.16,47.99,47.96
16,Austria,AUT,solar_electricity,0.00,0.01,0.01,0.01,0.02,0.02,0.02,0.02,0.03,0.05,0.09,0.17,0.34,0.63,0.79,0.94,1.10,1.27,1.46,1.70,2.04,2.78,3.79,6.89,8.14,10.54,11.24,12.77,14.30,15.83,17.36,19.00
17,Austria,AUT,solar_share_elec,0.00,0.02,0.02,0.02,0.03,0.03,0.03,0.03,0.05,0.08,0.14,0.29,0.52,1.03,1.35,1.62,1.78,1.99,2.36,2.52,3.09,4.39,6.19,10.14,10.78,14.74,14.99,16.59,18.11,19.56,20.94,19.39


In [25]:
# =====================================================================
# 1. INITIAL SETUP AND SCORE CALCULATION (from df_final)
# =====================================================================
df_gap = df_europe_target.copy()
col_2030 = 2030 if 2030 in df_gap.columns else "2030"

df_gap["gap_%"] = np.where(
    df_gap["2030_target"] == 0, 
    0,
    ((df_gap[col_2030] - df_gap["2030_target"]) / df_gap["2030_target"]) * 100
)
df_gap["gap_%"] = df_gap["gap_%"].replace([np.inf, -np.inf], 0).fillna(0)


# =====================================================================
# 2. SMART CLASSIFICATION & SCORE COMPUTATION
# =====================================================================
def classify_smart_gap(pred, target, metric, threshold=0.10):
    if pd.isna(target): 
        return "No target"
    if pd.isna(pred): 
        return "No data"
    
    is_fossil = any(word in str(metric).lower() for word in ["fossil", "coal", "gas", "oil"])
    
    if target == 0:
        if pred == 0:
            return "On Track"
        return "Failing" if is_fossil else "Exceeding Goal"
    
    gap = (pred - target) / target
    
    if abs(gap) <= threshold:
        return "On Track"
    
    if not is_fossil:
        return "Exceeding Goal" if gap > threshold else "Failing"
    else:
        return "Exceeding Goal" if gap < -threshold else "Failing"


df_gap["judgement"] = df_gap.apply(
    lambda r: classify_smart_gap(r[col_2030], r["2030_target"], r["energy_metric"]),
    axis=1
)


def calculate_smart_score(row):
    if row["judgement"] in ["No target", "No data"]:
        return np.nan
    
    if row["judgement"] in ["On Track", "Exceeding Goal"]:
        return 100.0
    
    is_fossil = any(word in str(row["energy_metric"]).lower() for word in ["fossil", "coal", "gas", "oil"])
    
    if not is_fossil:
        score = 100 + row["gap_%"]
    else:
        score = 100 - row["gap_%"]
    
    return max(0, min(100, score))


df_gap["achievement_score"] = df_gap.apply(calculate_smart_score, axis=1)

df_valid = df_gap[~df_gap["judgement"].isin(["No target", "No data"])].copy()


# =====================================================================
# 3. CATEGORY MAPPING
# =====================================================================
cat_mapping = {
    "solar": "Solar", "wind": "Wind", "hydro": "Hydro", "biofuel": "Biofuel",
    "nuclear": "Nuclear", "coal": "Coal", "gas": "Natural Gas", "oil": "Oil",
    "fossil": "Total Fossil", "renewables": "Total Renewables", "low_carbon": "Low Carbon"
}

def get_cat(metric):
    for key, name in cat_mapping.items():
        if key in str(metric).lower():
            return name
    return "Other"

df_valid["source_cat"] = df_valid["energy_metric"].apply(get_cat)


# =====================================================================
# 4. PLOTLY VISUAL SETTINGS (Predictive_Global style)
# =====================================================================
bg_color = "#D3D3D3"  # Light grey used in your screenshots
color_map = {"Exceeding Goal": "#2ca02c", "On Track": "#f1c40f", "Failing": "#d62728"}
statuses_ordered = ["Exceeding Goal", "On Track", "Failing"]

output_dir = r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

categories = ["All Categories"] + sorted(df_valid["source_cat"].unique().tolist())

In [26]:
# =====================================================================
# 5. DONUT CHART (EUROPE STATUS) - UPDATED ORDER & STYLE
# =====================================================================
import plotly.graph_objects as go
import pandas as pd
import os

fig_donut = go.Figure()

# Define the logical order of transition status
target_order = ["Exceeding Goal", "On Track", "Failing"]

for cat in categories:
    dff = df_valid if cat == "All Categories" else df_valid[df_valid['source_cat'] == cat]
    counts = dff['judgement'].value_counts().reset_index()
    counts.columns = ['judgement', 'count']
    
    # FORCING THE ORDER
    counts['judgement'] = pd.Categorical(counts['judgement'], categories=target_order, ordered=True)
    counts = counts.sort_values('judgement')
    
    fig_donut.add_trace(go.Pie(
        labels=counts['judgement'],
        values=counts['count'],
        hole=0.5,
        marker=dict(
            colors=[color_map.get(j, "#000") for j in counts['judgement']], 
            line=dict(color='#cccccc', width=1)
        ),
        name=cat,
        visible=(cat == "All Categories"),
        textinfo='label+percent',
        sort=False,             # IMPORTANT: prevents Plotly from reordering based on size
        direction='clockwise',  # Sets clockwise direction
        rotation=90             # Starts the first slice at 12 o'clock
    ))

# Build update buttons for each category
buttons_donut = [
    dict(
        label=cat,
        method="update",
        args=[{"visible": [c == cat for c in categories]},
              {"title.text": f"Global Transition Status - {cat}"}]
    )
    for cat in categories
]

fig_donut.update_layout(
    title=dict(
        text="Global Transition Status - All Categories",
        x=0.5, xanchor="center",
        font=dict(size=28, color="#383838")
    ),
  
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    
    legend=dict(
        orientation="v", 
        yanchor="top", y=1, 
        xanchor="left", x=1.02,
        font=dict(size=14, color="#383838")
    ),
    updatemenus=[dict(
        active=0, buttons=buttons_donut, 
        x=0.12, xanchor="left",
        y=1.15, yanchor="top",
        bgcolor="white", bordercolor="#4c4c4c", borderwidth=1
    )],
    annotations=[
        dict(
            text="<b>Filter:</b>",
            x=0.11, y=1.13,
            xref="paper", yref="paper",
            showarrow=False, xanchor="right", yanchor="middle",
            font=dict(size=14, color="#4c4c4c")
        )
    ],
    margin=dict(t=120)
)

# Show and Save
fig_donut.show()
fig_donut.write_html(os.path.join(output_dir, "23_europe_donut.html"))


In [27]:
# =====================================================================
# 6. STACKED BAR CHART (EUROPE) - UPDATED HEIGHT ONLY
# =====================================================================
import plotly.graph_objects as go
import os

fig_bar = go.Figure()

# Order for stacking: Bottom to Top (Failing -> On Track -> Exceeding)
target_order = ["Failing", "On Track", "Exceeding Goal"]

for cat in categories:
    dff = df_valid if cat == "All Categories" else df_valid[df_valid['source_cat'] == cat]
    grouped = dff.groupby(['energy_metric', 'judgement']).size().unstack(fill_value=0).reset_index()
    
    # Ensure all columns exist to avoid errors
    for s in target_order:
        if s not in grouped.columns:
            grouped[s] = 0
    
    # Adding traces in specific order for stacking
    for s in target_order: 
        fig_bar.add_trace(go.Bar(
            x=grouped['energy_metric'],
            y=grouped[s],
            name=s,
            marker_color=color_map[s],
            visible=(cat == "All Categories")
        ))

# Multi-category buttons (each category handles 3 traces)
buttons_bar = [
    dict(
        label=cat,
        method="update",
        args=[{"visible": [i//3 == idx for i in range(len(categories)*3)]},
              {"title.text": f"2030 Scorecard: Countries per Status ({cat})"}]
    )
    for idx, cat in enumerate(categories)
]

fig_bar.update_layout(
    height=1450, 
    barmode='stack',
    title=dict(
        text="2030 Scorecard: Countries per Status (All Categories)",
        x=0.5, xanchor="center",
        font=dict(size=28, color="#383838")
    ),
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    xaxis_tickangle=-45,
    
    legend=dict(
        orientation="v", 
        yanchor="top", y=1, 
        xanchor="left", x=1.02,
        traceorder="reversed", 
        font=dict(size=14, color="#383838")
    ),
    updatemenus=[dict(
        active=0, buttons=buttons_bar, 
        x=0.12, xanchor="left",
        y=1.15, yanchor="top",
        bgcolor="white", bordercolor="#4c4c4c", borderwidth=1
    )],
    annotations=[
        dict(
            text="<b>Filter:</b>",
            y=1.13,
            xref="paper", yref="paper",
            showarrow=False, xanchor="right", yanchor="middle",
            font=dict(size=14, color="#4c4c4c")
        )
    ],
    margin=dict(t=120, b=140)
)

# Show & Save
fig_bar.show()
fig_bar.write_html(os.path.join(output_dir, "24_europe_stacked_bar.html"))

In [28]:
# =====================================================================
# 7. BOX PLOT (with points) 
# =====================================================================
fig_box = go.Figure()

df_box_valid = df_valid[(df_valid["gap_%"] >= -100) & (df_valid["gap_%"] <= 300)].copy()
target_order = ["Exceeding Goal", "On Track", "Failing"]

for cat in categories:
    dff = df_box_valid if cat == "All Categories" else df_box_valid[df_box_valid['source_cat'] == cat]
    
    
    for s in target_order:
        dff_s = dff[dff['judgement'] == s]
        
        fig_box.add_trace(go.Box(
            x=dff_s['energy_metric'],
            y=dff_s['gap_%'],
            name=s,
            marker_color=color_map[s],
            visible=(cat == "All Categories"),
            boxpoints='all',
            jitter=0.3,
            pointpos=-1.8
        ))

buttons_box = [
    dict(
        label=cat,
        method="update",
        args=[{"visible": [i//3 == idx for i in range(len(categories)*3)]},
              {"title.text": f"2030 Target Gap Depth (%) - {cat}"}]
    )
    for idx, cat in enumerate(categories)
]

fig_box.update_layout(
    height=1450,  
    boxmode='group',
    title=dict(
        text="2030 Target Gap Depth (%) - All Categories",
        x=0.5, xanchor="center",
        font=dict(size=28, color="#383838")
    ),
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    xaxis_tickangle=-45,
    
    legend=dict(
        orientation="v", 
        yanchor="top", y=1, 
        xanchor="left", x=1.02,
        traceorder="normal", 
        font=dict(size=14, color="#383838")
    ),
    updatemenus=[dict(
        active=0, buttons=buttons_box, 
        x=0.12, xanchor="left",
        y=1.15, yanchor="top",
        bgcolor="white", bordercolor="#4c4c4c", borderwidth=1
    )],
    annotations=[
        dict(
            text="<b>Filter:</b>",
            y = 0.3,
            x = 1.0,
            xref="paper", yref="paper",
            showarrow=False, xanchor="right", yanchor="middle",
            font=dict(size=14, color="#4c4c4c")
        )
    ],
    margin=dict(t=120, b=140)
    
)

fig_box.add_hline(
    y=0, line_dash="dash", line_color="#333333",
     annotation_text="Perfect Target (0%)", annotation_position="top right")

# Save & Run
fig_box.write_html(os.path.join(output_dir, "25_europe_boxplot.html"))
fig_box.show()


In [29]:
# =====================================================================
# 8. RADAR CHART (Europe Total + Country Selector) - FILTER TOP-RIGHT
# =====================================================================
import plotly.graph_objects as go
import pandas as pd
import os

europe_avg = df_valid.groupby('energy_metric')['achievement_score'].mean().reset_index()
europe_avg['country'] = "Europe Total"

# Combine average data with individual country data
radar_combined = pd.concat([
    europe_avg,
    df_valid[['country', 'energy_metric', 'achievement_score']]
])

countries_list = ["Europe Total"] + sorted(df_valid['country'].unique())

fig_radar = go.Figure()

for country in countries_list:
    dff = radar_combined[radar_combined['country'] == country]
    
    metrics_present = dff['energy_metric'].tolist()
    scores_present = dff['achievement_score'].tolist()
    
    # Close the radar polygon
    if len(metrics_present) > 0:
        metrics_present.append(metrics_present[0])
        scores_present.append(scores_present[0])
    
    fig_radar.add_trace(go.Scatterpolar(
        r=scores_present,
        theta=metrics_present,
        fill='toself',
        name=country,
        visible=(country == "Europe Total"),
        marker=dict(color="#4169E1")
    ))

# Build update buttons for each country
buttons_radar = [
    dict(
        label=c,
        method="update",        
        args=[{"visible": [cnt == c for cnt in countries_list]},
              {"title.text": f"Target Achievement Profile: {c}"}] 
    )
    for c in countries_list
]

fig_radar.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 100], gridcolor="rgba(0,0,0,0.1)"),
        angularaxis=dict(gridcolor="rgba(0,0,0,0.1)")
    ),
    title=dict(
        text="Target Achievement Profile: Europe Total",
        x=0.5, xanchor="center",
        font=dict(size=28, color="#383838")
    ),
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    showlegend=False,
    
    # DROPDOWN POSITIONED TOP-RIGHT
    updatemenus=[dict(
        active=0, buttons=buttons_radar, 
        x=1.0, xanchor="right",
        y=1.15, yanchor="top",
        bgcolor="white", bordercolor="#4c4c4c", borderwidth=1
    )],
    # LABEL "COUNTRY:" ALIGNED TO DROPDOWN
    annotations=[
        dict(
            text="<b>Country:</b>",
            x=0.88, y=1.13,
            xref="paper", yref="paper",
            showarrow=False, xanchor="right", yanchor="middle",
            font=dict(size=14, color="#4c4c4c")
        )
    ],
    margin=dict(t=120)
)

# Show and Save
fig_radar.show()
fig_radar.write_html(os.path.join(output_dir, "26_europe_radar.html"))



In [30]:
# --- EUROPE CONFIGURATION ---
df = df_europe_target
title = "Europe Electricity Transition"
filename = r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\28_europe_target_area.html"

low_carbon_metrics = [
    "solar_electricity", "wind_electricity", "hydro_electricity",
    "nuclear_electricity", "biofuel_electricity", "other_renewable_electricity"
]

fossil_metrics = ["coal_electricity", "gas_electricity"]

color_map = {
    "solar_electricity": "#f1c40f",
    "wind_electricity": "#2ca02c",
    "hydro_electricity": "#1f77b4",
    "nuclear_electricity": "#9467bd",
    "biofuel_electricity": "#8c564b",
    "other_renewable_electricity": "#17becf",
    "coal_electricity": "#4c4c4c",
    "gas_electricity": "#C39BD3"
}


# 1. Data Preparation
year_cols = [c for c in df.columns if str(c).isdigit()]
df_melted = df.melt(
    id_vars=["energy_metric"],
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)

df_melted["year"] = df_melted["year"].astype(int)

df_yearly = (
    df_melted.groupby(["year", "energy_metric"])["value"]
    .sum()
    .reset_index()
)

df_plot = (
    df_yearly.pivot(index="year", columns="energy_metric", values="value")
    .reset_index()
    .sort_values("year")
)

# 2. Create Figure
fig_eu = go.Figure()

available_low = [c for c in low_carbon_metrics if c in df_plot.columns]
for col in available_low:
    fig_eu.add_trace(go.Scatter(
        x=df_plot["year"],
        y=df_plot[col],
        name=col.replace("_electricity", "").title(),
        mode="lines",
        stackgroup="one",
        line=dict(width=0.5),
        fillcolor=color_map.get(col),
        visible=True
    ))

available_fossil = [c for c in fossil_metrics if c in df_plot.columns]
for col in available_fossil:
    fig_eu.add_trace(go.Scatter(
        x=df_plot["year"],
        y=df_plot[col],
        name=col.replace("_electricity", "").title(),
        mode="lines",
        stackgroup="one",
        line=dict(width=0.5),
        fillcolor=color_map.get(col),
        visible=False
    ))

# 3. Layout and Filter Menu
fig_eu.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="Low-Carbon Mix",
                    method="update",
                    args=[
                        {"visible": [True]*len(available_low) + [False]*len(available_fossil)},
                        {"title": f"{title}: Low-Carbon Projection to 2030"}
                    ]
                ),
                dict(
                    label="Fossil Fuel Mix",
                    method="update",
                    args=[
                        {"visible": [False]*len(available_low) + [True]*len(available_fossil)},
                        {"title": f"{title}: Fossil Fuel Projection to 2030"}
                    ]
                )
            ],
            direction="down",
            x=0.01,
            xanchor="left",
            y=1.15,
            bgcolor="#FFF8E7",
            bordercolor="#383838",
            font=dict(size=14, color="#383838")
        )
    ],
    height=750,
    paper_bgcolor="#FFF8E7",
    plot_bgcolor="#FFF8E7",
    title=dict(
        text=f"{title}: Low-Carbon Projection to 2030",
        x=0.5,
        font=dict(size=24, color="#383838")
    ),
    xaxis=dict(
        title="Year",
        gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color="#383838")
    ),
    yaxis=dict(
        title="Electricity Generation (TWh)",
        gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color="#383838")
    ),
    font=dict(color="#383838"),
    margin=dict(t=120)
)

fig_eu.add_vline(
    x=2025,
    line_dash="dash",
    line_color="#383838",
    annotation_text="Prediction Start"
)

# Save
fig_eu.write_html(r"C:\Users\eirad\Desktop\IT ACADEMY\Sprint_13_Proyecto_Final\Quarto_Report\Plot\28_road_euro.html")
# Show
fig_eu.show()